In [4]:
import os
import requests
import zipfile

# Create data folder
os.makedirs("data", exist_ok=True)

# Dataset zip URL
url = "https://www.kaggle.com/api/v1/datasets/download/uciml/default-of-credit-card-clients-dataset"

# Headers (required by Kaggle)
headers = {
    "User-Agent": "Mozilla/5.0"
}

# Download zip
response = requests.get(url, headers=headers, stream=True)

zip_path = "data/credit_default.zip"
with open(zip_path, "wb") as f:
    for chunk in response.iter_content(chunk_size=1024):
        f.write(chunk)

# Unzip
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall("data")

print("Dataset downloaded and extracted!")


Dataset downloaded and extracted!


In [5]:
import pandas as pd

df = pd.read_csv("data/UCI_Credit_Card.csv")
df.head()


,ID,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_0,PAY_2,PAY_3,PAY_4,...,BILL_AMT4,BILL_AMT5,BILL_AMT6,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6,default.payment.next.month
0,1,20000.0,2,2,1,24,2,2,-1,-1,...,0.0,0.0,0.0,0.0,689.0,0.0,0.0,0.0,0.0,1
1,2,120000.0,2,2,2,26,-1,2,0,0,...,3272.0,3455.0,3261.0,0.0,1000.0,1000.0,1000.0,0.0,2000.0,1
2,3,90000.0,2,2,2,34,0,0,0,0,...,14331.0,14948.0,15549.0,1518.0,1500.0,1000.0,1000.0,1000.0,5000.0,0
3,4,50000.0,2,2,1,37,0,0,0,0,...,28314.0,28959.0,29547.0,2000.0,2019.0,1200.0,1100.0,1069.0,1000.0,0
4,5,50000.0,1,2,1,57,-1,0,-1,0,...,20940.0,19146.0,19131.0,2000.0,36681.0,10000.0,9000.0,689.0,679.0,0


In [17]:
rows, cols = df.shape
print("Total Rows:", rows)
print("Total Columns:", cols)



Total Rows: 30000
Total Columns: 25


In [10]:
df.isnull().sum().sum()

np.int64(0)

In [12]:
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30000 entries, 0 to 29999
Data columns (total 25 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   ID                          30000 non-null  int64  
 1   LIMIT_BAL                   30000 non-null  float64
 2   SEX                         30000 non-null  int64  
 3   EDUCATION                   30000 non-null  int64  
 4   MARRIAGE                    30000 non-null  int64  
 5   AGE                         30000 non-null  int64  
 6   PAY_0                       30000 non-null  int64  
 7   PAY_2                       30000 non-null  int64  
 8   PAY_3                       30000 non-null  int64  
 9   PAY_4                       30000 non-null  int64  
 10  PAY_5                       30000 non-null  int64  
 11  PAY_6                       30000 non-null  int64  
 12  BILL_AMT1                   30000 non-null  float64
 13  BILL_AMT2                   300

# DATA PREPROCESSING + LOGISTIC REGRESSION

In [6]:
# Drop ID column
df = df.drop(columns=['ID'])

# Define target
target = 'default.payment.next.month'

X = df.drop(columns=[target])
y = df[target]


In [7]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)


In [8]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


In [9]:
from sklearn.linear_model import LogisticRegression

log_reg = LogisticRegression(max_iter=1000, random_state=42)
log_reg.fit(X_train_scaled, y_train)


LogisticRegression(max_iter=1000, random_state=42)

In [10]:
y_pred = log_reg.predict(X_test_scaled)
y_prob = log_reg.predict_proba(X_test_scaled)[:, 1]


In [11]:
from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    matthews_corrcoef
)

accuracy = accuracy_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_prob)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
mcc = matthews_corrcoef(y_test, y_pred)

print("Logistic Regression Metrics")
print("Accuracy:", accuracy)
print("AUC:", auc)
print("Precision:", precision)
print("Recall:", recall)
print("F1 Score:", f1)
print("MCC:", mcc)


Logistic Regression Metrics
Accuracy: 0.8076666666666666
AUC: 0.7076355036089734
Precision: 0.6868250539956804
Recall: 0.23963828183873398
F1 Score: 0.3553072625698324
MCC: 0.32444311464457076


In [12]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, y_pred)
cm


array([[4528,  145],
       [1009,  318]])

# DECISION TREE

In [13]:
from sklearn.tree import DecisionTreeClassifier

dt_model = DecisionTreeClassifier(
    max_depth=10,
    random_state=42
)

dt_model.fit(X_train, y_train)


DecisionTreeClassifier(max_depth=10, random_state=42)

In [14]:
y_pred_dt = dt_model.predict(X_test)
y_prob_dt = dt_model.predict_proba(X_test)[:, 1]


In [15]:
accuracy_dt = accuracy_score(y_test, y_pred_dt)
auc_dt = roc_auc_score(y_test, y_prob_dt)
precision_dt = precision_score(y_test, y_pred_dt)
recall_dt = recall_score(y_test, y_pred_dt)
f1_dt = f1_score(y_test, y_pred_dt)
mcc_dt = matthews_corrcoef(y_test, y_pred_dt)

print("Decision Tree Metrics")
print("Accuracy:", accuracy_dt)
print("AUC:", auc_dt)
print("Precision:", precision_dt)
print("Recall:", recall_dt)
print("F1 Score:", f1_dt)
print("MCC:", mcc_dt)


Decision Tree Metrics
Accuracy: 0.8106666666666666
AUC: 0.7222772485591601
Precision: 0.6264900662251656
Recall: 0.35644310474755087
F1 Score: 0.4543707973102786
MCC: 0.37052737913003236


In [16]:
cm_dt = confusion_matrix(y_test, y_pred_dt)
cm_dt


array([[4391,  282],
       [ 854,  473]])

# K-NEAREST NEIGHBORS (KNN)

In [17]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    matthews_corrcoef,
    confusion_matrix
)


In [18]:
from sklearn.neighbors import KNeighborsClassifier

knn_model = KNeighborsClassifier(n_neighbors=5)

knn_model.fit(X_train_scaled, y_train)


KNeighborsClassifier()

In [19]:
import os
os.environ["LOKY_MAX_CPU_COUNT"] = "4"  # or your CPU count


In [20]:
y_pred_knn = knn_model.predict(X_test_scaled)
y_prob_knn = knn_model.predict_proba(X_test_scaled)[:, 1]

In [21]:
accuracy_knn = accuracy_score(y_test, y_pred_knn)
auc_knn = roc_auc_score(y_test, y_prob_knn)
precision_knn = precision_score(y_test, y_pred_knn)
recall_knn = recall_score(y_test, y_pred_knn)
f1_knn = f1_score(y_test, y_pred_knn)
mcc_knn = matthews_corrcoef(y_test, y_pred_knn)

print("KNN Metrics")
print("Accuracy:", accuracy_knn)
print("AUC:", auc_knn)
print("Precision:", precision_knn)
print("Recall:", recall_knn)
print("F1 Score:", f1_knn)
print("MCC:", mcc_knn)


KNN Metrics
Accuracy: 0.7928333333333333
AUC: 0.701390130833851
Precision: 0.548723897911833
Recall: 0.35644310474755087
F1 Score: 0.4321608040201005
MCC: 0.3232672232965524


In [22]:
cm_knn = confusion_matrix(y_test, y_pred_knn)
cm_knn


array([[4284,  389],
       [ 854,  473]])

# NAIVE BAYES (GaussianNB)

In [23]:
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    matthews_corrcoef,
    confusion_matrix
)


In [24]:
nb_model = GaussianNB()
nb_model.fit(X_train, y_train)


GaussianNB()

In [25]:
y_pred_nb = nb_model.predict(X_test)
y_prob_nb = nb_model.predict_proba(X_test)[:, 1]


In [26]:
accuracy_nb = accuracy_score(y_test, y_pred_nb)
auc_nb = roc_auc_score(y_test, y_prob_nb)
precision_nb = precision_score(y_test, y_pred_nb)
recall_nb = recall_score(y_test, y_pred_nb)
f1_nb = f1_score(y_test, y_pred_nb)
mcc_nb = matthews_corrcoef(y_test, y_pred_nb)

print("Naive Bayes Metrics")
print("Accuracy:", accuracy_nb)
print("AUC:", auc_nb)
print("Precision:", precision_nb)
print("Recall:", recall_nb)
print("F1 Score:", f1_nb)
print("MCC:", mcc_nb)


Naive Bayes Metrics
Accuracy: 0.416
AUC: 0.6515671244531792
Precision: 0.249597423510467
Recall: 0.8176337603617182
F1 Score: 0.3824462460345435
MCC: 0.1110873771184327


In [27]:
cm_nb = confusion_matrix(y_test, y_pred_nb)
cm_nb


array([[1411, 3262],
       [ 242, 1085]])

# RANDOM FOREST CLASSIFIER (Ensemble)

In [28]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    matthews_corrcoef,
    confusion_matrix
)


In [29]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(
    n_estimators=30,   # reduced from 100
    max_depth=10,
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train, y_train)


RandomForestClassifier(max_depth=10, n_estimators=30, n_jobs=-1,
                       random_state=42)

In [30]:
y_pred_rf = rf_model.predict(X_test)
y_prob_rf = rf_model.predict_proba(X_test)[:, 1]


In [31]:
accuracy_rf = accuracy_score(y_test, y_pred_rf)
auc_rf = roc_auc_score(y_test, y_prob_rf)
precision_rf = precision_score(y_test, y_pred_rf)
recall_rf = recall_score(y_test, y_pred_rf)
f1_rf = f1_score(y_test, y_pred_rf)
mcc_rf = matthews_corrcoef(y_test, y_pred_rf)

print("Random Forest (Reduced Size) Metrics")
print("Accuracy :", accuracy_rf)
print("AUC      :", auc_rf)
print("Precision:", precision_rf)
print("Recall   :", recall_rf)
print("F1 Score :", f1_rf)
print("MCC      :", mcc_rf)

Random Forest (Reduced Size) Metrics
Accuracy : 0.8173333333333334
AUC      : 0.769426442625798
Precision: 0.6629055007052186
Recall   : 0.3541823662396383
F1 Score : 0.46168958742632615
MCC      : 0.3896168365252773


In [32]:
cm_rf = confusion_matrix(y_test, y_pred_rf)
cm_rf


array([[4434,  239],
       [ 857,  470]])

# XGBOOST CLASSIFIER (Ensemble – Boosting)


In [33]:
pip install xgboost


Note: you may need to restart the kernel to use updated packages.


In [34]:
from xgboost import XGBClassifier
from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    matthews_corrcoef,
    confusion_matrix
)


In [35]:
xgb_model = XGBClassifier(
    n_estimators=100,
    max_depth=5,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric='logloss',
    random_state=42
)

xgb_model.fit(X_train, y_train)


XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='logloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.1, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=5, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=100, n_jobs=None,
              num_parallel_tree=None, ...)

In [36]:
y_pred_xgb = xgb_model.predict(X_test)
y_prob_xgb = xgb_model.predict_proba(X_test)[:, 1]


In [37]:
accuracy_xgb = accuracy_score(y_test, y_pred_xgb)
auc_xgb = roc_auc_score(y_test, y_prob_xgb)
precision_xgb = precision_score(y_test, y_pred_xgb)
recall_xgb = recall_score(y_test, y_pred_xgb)
f1_xgb = f1_score(y_test, y_pred_xgb)
mcc_xgb = matthews_corrcoef(y_test, y_pred_xgb)

print("XGBoost Metrics")
print("Accuracy:", accuracy_xgb)
print("AUC:", auc_xgb)
print("Precision:", precision_xgb)
print("Recall:", recall_xgb)
print("F1 Score:", f1_xgb)
print("MCC:", mcc_xgb)


XGBoost Metrics
Accuracy: 0.8188333333333333
AUC: 0.7783584480809846
Precision: 0.6657458563535912
Recall: 0.3632253202712886
F1 Score: 0.47001462701121405
MCC: 0.3968111967310267


In [38]:
cm_xgb = confusion_matrix(y_test, y_pred_xgb)
cm_xgb


array([[4431,  242],
       [ 845,  482]])

# METRIC COMPARISON TABLE

In [39]:
import pandas as pd

results = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "Decision Tree",
        "KNN",
        "Naive Bayes",
        "Random Forest",
        "XGBoost"
    ],
    "Accuracy": [
        accuracy,
        accuracy_dt,
        accuracy_knn,
        accuracy_nb,
        accuracy_rf,      # UPDATED RF
        accuracy_xgb
    ],
    "AUC": [
        auc,
        auc_dt,
        auc_knn,
        auc_nb,
        auc_rf,           # UPDATED RF
        auc_xgb
    ],
    "Precision": [
        precision,
        precision_dt,
        precision_knn,
        precision_nb,
        precision_rf,     # UPDATED RF
        precision_xgb
    ],
    "Recall": [
        recall,
        recall_dt,
        recall_knn,
        recall_nb,
        recall_rf,        # UPDATED RF
        recall_xgb
    ],
    "F1 Score": [
        f1,
        f1_dt,
        f1_knn,
        f1_nb,
        f1_rf,            # UPDATED RF
        f1_xgb
    ],
    "MCC": [
        mcc,
        mcc_dt,
        mcc_knn,
        mcc_nb,
        mcc_rf,           # UPDATED RF
        mcc_xgb
    ]
})

results


,Model,Accuracy,AUC,Precision,Recall,F1 Score,MCC
0,Logistic Regression,0.807667,0.707636,0.686825,0.239638,0.355307,0.324443
1,Decision Tree,0.810667,0.722277,0.626490,0.356443,0.454371,0.370527
2,KNN,0.792833,0.701390,0.548724,0.356443,0.432161,0.323267
3,Naive Bayes,0.416000,0.651567,0.249597,0.817634,0.382446,0.111087
4,Random Forest,0.817333,0.769426,0.662906,0.354182,0.461690,0.389617
5,XGBoost,0.818833,0.778358,0.665746,0.363225,0.470015,0.396811


In [40]:
results.to_csv("model_comparison_metrics_table.csv", index=False)


In [41]:
import os

os.makedirs("model", exist_ok=True)


In [42]:
import joblib

joblib.dump(log_reg, "model/logistic_model.pkl")
joblib.dump(dt_model, "model/decision_tree_model.pkl")
joblib.dump(knn_model, "model/knn_model.pkl")
joblib.dump(nb_model, "model/naive_bayes_model.pkl")
joblib.dump(rf_model, "model/random_forest_model.pkl")
joblib.dump(xgb_model, "model/xgboost_model.pkl")

# Save scaler
joblib.dump(scaler, "model/scaler.pkl")


['model/scaler.pkl']

In [43]:
joblib.dump(rf_model, "model/random_forest_model.pkl")


['model/random_forest_model.pkl']

In [44]:
import pandas as pd

# Combine X_test and y_test
test_df = X_test.copy()
test_df["default.payment.next.month"] = y_test.values

# OPTIONAL: add ID column (nice for realism)
test_df.insert(0, "ID", range(1, len(test_df) + 1))

# Save clean CSV
test_df.to_csv("test_data.csv", index=False)

print("✅ test_data.csv generated successfully!")



✅ test_data.csv generated successfully!
